In [ ]:
%load_ext autoreload
%autoreload 2

import xarray as xr
import torch
import yaml
import sys
from pathlib import Path
root_dir = Path.cwd().parent   
sys.path.append(str(root_dir))

from data.dataset import ERA5Dataset
from data.dataloader import *

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version (runtime):", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

from huggingface_hub import snapshot_download

Torch version: 2.3.1
CUDA available: True
CUDA version (runtime): 12.1
GPU: Quadro T1000 with Max-Q Design


c:\Users\yaren\miniconda3\envs\weather-cast\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Load the configurations and the dataset.  

In [ ]:
# Load config
config_path = Path.cwd().parent / "utils" / "default_config.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)
print("Config loaded!")

snapshot_download(repo_id=config["data"]["repo_id"],
                  repo_type='dataset',
                  local_dir=config["data"]["local_dir"], 
                  allow_patterns="*")

Config loaded!


Get the dataloaders for each split, ready to plug into the ML pipeline.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

train_loader, val_loader, test_loader = get_dataloaders(
    config=config["data"],
    batch_size=config["training"]["batch_size"], 
    num_workers=config["training"]["num_workers"],
    device=device
)
print("\nDataloaders ready!")

Using device: cuda
Creating datasets...
Dataset sizes -> Train: 87400, Val: 17488, Test: 17488
Creating dataloaders...

Dataloaders ready!


Test out the first iteration.

In [4]:
X, y = next(iter(train_loader)) 
print("Device:", X.device, y.device)

# sample = train_loader.dataset[0]
# print(type(sample), len(sample))
# print(sample)

Device: cuda:0 cuda:0
